# Chapter 6: Moduli Spaces of Stable Maps

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 6, printed pp. 153-200; PDF pp. 168-215. Sections: 6.1-6.7.

    ## Chapter Goal

    Simple stable maps, transversality, evaluation maps, semipositivity, pseudocycles, Gromov-Witten pseudocycles, and graph pseudocycles.

    The guiding question is:

    > When does the compactified stable-map space define a pseudocycle rather than a space with dangerous boundary?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| simple stable map | decorated tree with simple nonconstant vertices | stratum regularity |
| evaluation map | marked-point image map | rank against constraints |
| semipositivity | index rule excluding bad low-codimension bubbles | codimension ledger |
| pseudocycle | map with small limiting boundary | boundary codimension > 1 |
| graph pseudocycle | parameterized comparison map | cobordism check |


## Standalone Reading Guide

    After stable maps have been compactified, the next question is whether their evaluation images can be used as cycles. The answer is subtle because the compactification has boundary strata corresponding to bubbles and nodes. A pseudocycle is allowed to have limiting behavior, but that limiting image must have sufficiently high codimension so it does not affect intersection numbers with generic constraints.

The chapter uses simplicity and semipositivity to control this situation. Simple stable maps give transversality a usable target on each stratum. Semipositivity rules out troublesome sphere bubbles whose index would place them in codimension one. Evaluation maps then turn the moduli spaces into geometric representatives suitable for intersection theory. The pseudocycle of graphs is a way to compare choices and show that the resulting counts do not depend on auxiliary data.

The visualization here is a dimension budget. The main stratum carries the expected dimension. Each node and bubble type changes the budget. The check is not a proof of semipositivity, but it makes the goal transparent: all limiting evaluation images that are not part of the intended moduli problem must be too small to change the intersection number.

    ## Topics In This Notebook

    - simple stable maps and transversality over strata
- evaluation maps with marked-point constraints
- semipositivity as a dimension-control hypothesis
- pseudocycle boundary codimension
- graph constructions for comparing moduli data

    ## Visualization Storyboard

    - A stratum-codimension chart shows why semipositivity keeps bubbling boundaries from corrupting counts.
- A dependency graph links stable-map compactification to pseudocycle output.
- A ledger compares main, one-node, and two-node strata.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-06'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['simple stable map', 'evaluation map', 'semipositivity', 'pseudocycle', 'graph pseudocycle']
CONCEPT_EDGES = [('simple stable map', 'evaluation map'), ('evaluation map', 'semipositivity'), ('semipositivity', 'pseudocycle'), ('pseudocycle', 'graph pseudocycle')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=38, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Moduli Spaces of Stable Maps')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '153-200',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Moduli Spaces of Stable Maps. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
strata = ["main stratum", "one node", "vertical bubble", "two nodes"]
expected_dim = np.array([8, 6, 4, 2])
boundary_codim = expected_dim[0] - expected_dim
safe = boundary_codim > 1

fig, ax = plt.subplots(figsize=(7.4, 4.8))
bars = ax.bar(strata, boundary_codim, color=["#6baed6" if ok else "#fb6a4a" for ok in safe])
ax.axhline(2, color="black", linestyle="--", linewidth=1, label="pseudocycle-safe threshold")
ax.set_ylabel("codimension in evaluation image model")
ax.set_title("Boundary codimension ledger for stable-map pseudocycles")
ax.tick_params(axis="x", rotation=20)
ax.legend()
for bar, codim in zip(bars, boundary_codim):
    ax.text(bar.get_x() + bar.get_width() / 2, codim + 0.05, str(int(codim)), ha="center")
fig_path = save_matplotlib(fig, UNIT, "figures", "pseudocycle-boundary-codimension.png")
plt.close(fig)

check = {
    "strata": strata,
    "expected_dimensions": expected_dim.tolist(),
    "boundary_codimensions": boundary_codim.tolist(),
    "safe_boundary_flags": safe.tolist(),
    "passed": bool(np.all(safe[1:])),
}
check_path = save_json(check, UNIT, "checks", "pseudocycle-codimension-checks.json")
display_artifact(fig_path, width=720)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'simple stable map', 'computational_object': 'decorated tree with simple nonconstant vertices', 'check': 'stratum regularity'}, {'item': 'evaluation map', 'computational_object': 'marked-point image map', 'check': 'rank against constraints'}, {'item': 'semipositivity', 'computational_object': 'index rule excluding bad low-codimension bubbles', 'check': 'codimension ledger'}, {'item': 'pseudocycle', 'computational_object': 'map with small limiting boundary', 'check': 'boundary codimension > 1'}, {'item': 'graph pseudocycle', 'computational_object': 'parameterized comparison map', 'check': 'cobordism check'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Lower one boundary codimension in the ledger to one. The pseudocycle check will fail, illustrating exactly why a codimension-one boundary would threaten invariance.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Pseudocycles allow compactification boundaries only when their images are small enough.
- Semipositivity is a dimension-control condition for bubble strata.
- Evaluation maps are the geometric carriers of Gromov-Witten constraints.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'pseudocycle-boundary-codimension.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'pseudocycle-codimension-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'pseudocycle-codimension-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
